In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from app.rules import check_rules

scam_post = """URGENT HIRING! Work from home data entry job. Earn Rs 3000 per day.
No interview required, direct joining. Pay registration fee of Rs 499 to confirm your seat.
Limited seats available. Send your Aadhaar card and bank account details
on WhatsApp 9876543210 or email hrjobs@gmail.com"""

result = check_rules(scam_post)

print("RULE SCORE:", result["rule_score"], "/ 60\n")
for r in result["triggered_rules"]:
    print(f"[+{r['weight']}] {r['rule']}")
    print(f"      matched: '{r['matched_text']}'")
    print(f"      why: {r['explanation']}\n")

RULE SCORE: 60 / 60

[+25] Upfront fee requested
      matched: 'registration fee'
      why: Asks you to pay money. Genuine employers never charge candidates to get a job.

[+15] Sensitive personal data requested
      matched: 'Aadhaar'
      why: Mentions Aadhaar, PAN, bank details, OTP, or PINs. Never share these before a verified offer letter.

[+12] No interview / instant selection
      matched: 'No interview'
      why: Skips the interview process. Legitimate companies assess candidates before hiring.

[+12] Unrealistic earnings
      matched: 'Earn Rs 3000 per day'
      why: Promises easy daily or weekly earnings, a common work-from-home scam pattern.

[+10] Personal contact instead of official channel
      matched: 'WhatsApp'
      why: Uses WhatsApp, Telegram, a personal email, or a mobile number instead of an official company channel.

[+8] Pressure / urgency tactics
      matched: 'URGENT HIRING'
      why: Creates urgency so you act before thinking or verifying.



In [2]:
genuine_post = """Software Engineer - Python. Infosys is hiring for its Bengaluru office.
Requirements: Bachelor's degree in Computer Science, 2+ years of experience with
Python and SQL. Selection process: online test, technical interview, and HR round.
Apply only through our official careers portal. Benefits include health insurance
and paid leave."""

result = check_rules(genuine_post)

print("RULE SCORE:", result["rule_score"], "/ 60")
print("Rules triggered:", len(result["triggered_rules"]))

RULE SCORE: 0 / 60
Rules triggered: 0


In [7]:
import pandas as pd
from collections import Counter

df = pd.read_csv("../data/cleaned_job_postings.csv")

results = df["combined_text"].apply(check_rules)
df["rule_score"] = results.apply(lambda r: r["rule_score"])

print("% of posts with rule score > 0:")
print((df.groupby("fraudulent")["rule_score"].apply(lambda s: (s > 0).mean() * 100)).round(1))

print("\nAverage rule score:")
print(df.groupby("fraudulent")["rule_score"].mean().round(2))

print("\nMost common rules on REAL posts:")
real_rules = Counter(r["rule"] for res in results[df["fraudulent"] == 0] for r in res["triggered_rules"])
for rule, count in real_rules.most_common():
    print(f"  {count:5d}  {rule}")

% of posts with rule score > 0:
fraudulent
0    2.7
1    2.1
Name: rule_score, dtype: float64

Average rule score:
fraudulent
0    0.29
1    0.27
Name: rule_score, dtype: float64

Most common rules on REAL posts:
    305  Personal contact instead of official channel
     51  Sensitive personal data requested
     23  Guaranteed job or income
      9  Pressure / urgency tactics
      7  Task-based scam pattern
      6  Unrealistic earnings
      3  Upfront fee requested
      1  No interview / instant selection


In [8]:
for rule_name in ["Personal contact instead of official channel", "Sensitive personal data requested"]:
    matches = Counter(
        r["matched_text"].lower()
        for res in results[df["fraudulent"] == 0]
        for r in res["triggered_rules"]
        if r["rule"] == rule_name
    )
    print(f"=== {rule_name} ===")
    for text, count in matches.most_common(12):
        print(f"  {count:4d}  '{text}'")
    print()

=== Personal contact instead of official channel ===
   108  '6774719060'
    53  'whatsapp'
    35  '7840115600'
     5  '8200005268'
     5  '7660307687'
     5  '8387427193'
     4  '8302020126'
     3  '9993864408'
     3  '9322166019'
     3  '7175568227'
     3  '8282636927'
     3  '8131207082'

=== Sensitive personal data requested ===
    48  'bank account'
     2  'otp'
     1  'pan card'



In [9]:
import importlib
import app.rules
importlib.reload(app.rules)
from app.rules import check_rules

# 1) Re-check the scam post
print("Scam post score:", check_rules(scam_post)["rule_score"], "/ 60")
print("Genuine post score:", check_rules(genuine_post)["rule_score"], "/ 60\n")

# 2) Re-check the whole dataset
results = df["combined_text"].apply(check_rules)
df["rule_score"] = results.apply(lambda r: r["rule_score"])

print("% of posts with rule score > 0:")
print((df.groupby("fraudulent")["rule_score"].apply(lambda s: (s > 0).mean() * 100)).round(1))

print("\nMost common rules on REAL posts:")
real_rules = Counter(r["rule"] for res in results[df["fraudulent"] == 0] for r in res["triggered_rules"])
for rule, count in real_rules.most_common():
    print(f"  {count:5d}  {rule}")

Scam post score: 60 / 60
Genuine post score: 0 / 60

% of posts with rule score > 0:
fraudulent
0    0.6
1    1.4
Name: rule_score, dtype: float64

Most common rules on REAL posts:
     35  Personal contact instead of official channel
     23  Guaranteed job or income
      9  Pressure / urgency tactics
      7  Task-based scam pattern
      6  Unrealistic earnings
      3  Upfront fee requested
      3  Sensitive personal data requested
      1  No interview / instant selection


In [10]:
import joblib

model = joblib.load("../models/scamshield_model.pkl")

print("Model loaded successfully!")
print("Steps inside the model:", [name for name, _ in model.steps])
print("Vocabulary size:", len(model.named_steps["tfidf"].vocabulary_))

Model loaded successfully!
Steps inside the model: ['tfidf', 'clf']
Vocabulary size: 30000


In [11]:
for label, text in [("SCAM POST", scam_post), ("GENUINE POST", genuine_post)]:
    prob = model.predict_proba([text])[0][1]
    pred = model.predict([text])[0]
    print(f"=== {label} ===")
    print(f"Scam probability: {prob:.2%}")
    print(f"Model verdict:    {'FAKE' if pred == 1 else 'REAL'}\n")

=== SCAM POST ===
Scam probability: 87.22%
Model verdict:    FAKE

=== GENUINE POST ===
Scam probability: 25.40%
Model verdict:    REAL



In [12]:
import numpy as np

vec = model.named_steps["tfidf"]
clf = model.named_steps["clf"]
words = vec.get_feature_names_out()

def explain(text, n=8):
    x = vec.transform([text])
    contrib = x.multiply(clf.coef_[0]).tocsr()
    idx, vals = contrib.indices, contrib.data
    order = np.argsort(vals)

    print("  Pushing towards FAKE:")
    for i in order[::-1][:n]:
        if vals[i] > 0:
            print(f"    +{vals[i]:.3f}  {words[idx[i]]}")
    print("  Pushing towards REAL:")
    for i in order[:n]:
        if vals[i] < 0:
            print(f"    {vals[i]:.3f}  {words[idx[i]]}")

print("=== SCAM POST ===")
explain(scam_post)
print("\n=== GENUINE POST ===")
explain(genuine_post)

=== SCAM POST ===
  Pushing towards FAKE:
    +0.686  data entry
    +0.534  earn
    +0.404  entry
    +0.392  work home
    +0.205  urgent
    +0.199  pay
    +0.195  email
    +0.185  send
  Pushing towards REAL:
    -0.125  bank
    -0.080  interview
    -0.048  joining
    -0.040  details
    -0.037  gmail
    -0.035  seat
    -0.034  account
    -0.031  day interview

=== GENUINE POST ===
  Pushing towards FAKE:
    +0.182  apply
    +0.167  requirements bachelor
    +0.146  requirements
    +0.122  hiring
    +0.121  benefits
    +0.099  paid
    +0.086  computer
    +0.081  sql
  Pushing towards REAL:
    -0.262  python
    -0.191  software
    -0.128  software engineer
    -0.116  computer science
    -0.099  health insurance
    -0.081  interview
    -0.072  science
    -0.069  official


In [13]:
from app.risk_engine import load_model, analyze_job_post

engine_model = load_model()

def show(result):
    print(f"RISK SCORE: {result['risk_score']} / 100  ->  {result['risk_label']}")
    print(f"  ML probability: {result['ml_probability']:.2%}  (ML score: {result['ml_score']} / 40)")
    print(f"  Rule score:     {result['rule_score']} / 60")
    print("\n  Why:")
    for e in result["explanations"]:
        print("   -", e)
    print("\n  Safety advice:")
    for a in result["safety_advice"]:
        print("   -", a)

print("=========== SCAM POST ===========")
show(analyze_job_post(scam_post, engine_model))

print("\n=========== GENUINE POST ===========")
show(analyze_job_post(genuine_post, engine_model))

=========== SCAM POST ===========
RISK SCORE: 95 / 100  ->  HIGH RISK
  ML probability: 87.22%  (ML score: 34.9 / 40)
  Rule score:     60 / 60

  Why:
   - Upfront fee requested (found: "registration fee"): Asks you to pay money. Genuine employers never charge candidates to get a job.
   - Sensitive personal data requested (found: "Aadhaar"): Mentions Aadhaar, PAN, bank details, OTP, or PINs. Never share these before a verified offer letter.
   - No interview / instant selection (found: "No interview"): Skips the interview process. Legitimate companies assess candidates before hiring.
   - Unrealistic earnings (found: "Earn Rs 3000 per day"): Promises easy daily or weekly earnings, a common work-from-home scam pattern.
   - Personal contact instead of official channel (found: "@gmail.com"): Uses WhatsApp, Telegram, a personal email, or a mobile number instead of an official company channel.
   - Pressure / urgency tactics (found: "URGENT HIRING"): Creates urgency so you act before thi

In [14]:
quiet_scam = df[df["fraudulent"] == 1]["combined_text"].iloc[0]

print("First 200 characters:", quiet_scam[:200], "...\n")
show(analyze_job_post(quiet_scam, engine_model))

First 200 characters: IC&E Technician Staffing & Recruiting done right for the Oil & Energy Industry!Represented candidates are automatically granted the following perks: Expert negotiations on your behalf, maximizing your ...

RISK SCORE: 38 / 100  ->  LOW RISK
  ML probability: 95.81%  (ML score: 38.3 / 40)
  Rule score:     0 / 60

  Why:
   - The ML model finds this post's language strongly similar to known fake job posts (96%).

  Safety advice:
   - No strong scam signals found, but always verify the employer.
   - Apply through the company's official careers page where possible.
   - Never pay any fee to get a job.


In [15]:
import importlib
import app.risk_engine
importlib.reload(app.risk_engine)
from app.risk_engine import analyze_job_post

for name, text in [("Indian scam", scam_post), ("Genuine", genuine_post), ("IC&E quiet scam", quiet_scam)]:
    r = analyze_job_post(text, engine_model)
    print(f"{name:<16} -> {r['risk_score']:>3} / 100  {r['risk_label']}")

print("\n=========== IC&E QUIET SCAM (full) ===========")
show(analyze_job_post(quiet_scam, engine_model))

Indian scam      ->  95 / 100  HIGH RISK
Genuine          ->  25 / 100  LOW RISK
IC&E quiet scam  ->  96 / 100  HIGH RISK

=========== IC&E QUIET SCAM (full) ===========
RISK SCORE: 96 / 100  ->  HIGH RISK
  ML probability: 95.81%  (ML score: 38.3 / 40)
  Rule score:     0 / 60

  Why:
   - The ML model finds this post's language strongly similar to known fake job posts (96%).

  Safety advice:
   - Do NOT pay any registration, training, or security fee.
   - Do NOT share Aadhaar, PAN, bank details, OTP, or UPI PIN.
   - Verify the company on its official website and LinkedIn page.
   - Report suspected fraud at cybercrime.gov.in or call the 1930 helpline.
